# Topic Analysis Notebook - Ran in Google Collab

The following notebook contains the cell outputs after running the code.


In [1]:
!pip install transformers==4.57.6
!pip install simpletransformers==0.70.5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.16.1
    Uninstalling huggingface_hub-1.16.1:
      Successfully uninstalled huggingface_hub-1.16.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.8/330.8 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.metrics import classification_report
from simpletransformers.classification import ClassificationModel, ClassificationArgs
import matplotlib.pyplot as plt
import seaborn as sn

# Custom Dataset Generation

In [3]:
imdb_path = 'topic-datasets/IMDB Dataset.csv'
kindle_path = 'topic-datasets/preprocessed_kindle_review .csv'

# Load datasets
imdb_full = pd.read_csv(imdb_path)
kindle_full = pd.read_csv(kindle_path)
restaurant_full = pd.read_csv('topic-datasets/Restaurant_Reviews.tsv', sep='\t')
test_raw = pd.read_csv('topic-datasets/Sentiment-topic-test.tsv', sep='\t')
topic_mapping = { 'movie': 0, 'book': 1, 'restaurant': 2}

# Sample exactly 1000 random instances reproducibly for each class
imdb_sample = imdb_full.sample(n=1000, random_state=42)
kindle_sample = kindle_full.sample(n=1000, random_state=42)
restaurant_sample = restaurant_full.sample(n=1000, random_state=42)

#Keep relevant columns for topic classification task
imdb_clean = pd.DataFrame({'text': imdb_sample['review'],  'labels': 0 })
kindle_clean = pd.DataFrame({ 'text': kindle_sample['reviewText'], 'labels': 1 })
restaurant_clean = pd.DataFrame({ 'text': restaurant_sample['Review'], 'labels': 2 })

all_training_data = pd.concat([imdb_clean, kindle_clean, restaurant_clean], ignore_index=True)
train = all_training_data.sample(frac=1, random_state=42).reset_index(drop=True)
train = train.dropna().reset_index(drop=True)

print(train.head())
print("\nClass distribution:")
print(train['labels'].value_counts())


                                                text  labels
0  This is a pretty good story,has a little of ev...       1
1  This is not a book to be set aside lightly. Th...       1
2  This is one of the most dissapointing purchase...       1
3  Alexander Nevsky (1938) is a brilliant piece o...       0
4  Insults, profound deuchebaggery, and had to go...       2

Class distribution:
labels
1    1000
0    1000
2    1000
Name: count, dtype: int64


# Converting Dataset to Compatible (Simpletransformers) Format


In [4]:
from sklearn.model_selection import train_test_split

test = pd.DataFrame({ 'text': test_raw['text'], 'labels': test_raw['topic'].map(topic_mapping)})
train, dev = train_test_split(train, test_size=0.1, random_state=42, stratify=train['labels'])

print(test.head())


                                                text  labels
0  It took eight years for Warner Brothers to rec...       0
1  All the New York University students love this...       2
2  This Italian place is really trendy but they h...       2
3  In conclusion, my review of this book would be...       1
4  The story of this movie is focused on Carl Bra...       0


# Model Parameters and Training


In [5]:
model_args = ClassificationArgs()
model_args.overwrite_output_dir=True
model_args.evaluate_during_training=True
model_args.num_train_epochs=10
model_args.train_batch_size=32
model_args.learning_rate=4e-6
model_args.max_seq_length=256
model_args.use_early_stopping=True
model_args.early_stopping_delta=0.01
model_args.early_stopping_metric='eval_loss'
model_args.early_stopping_metric_minimize=True
model_args.early_stopping_patience=2
model_args.evaluate_during_training_steps=32

steps_per_epoch = int(np.ceil(len(train) / float(model_args.train_batch_size)))
print('Each epoch will have {:,} steps.'.format(steps_per_epoch))

Each epoch will have 85 steps.


In [6]:
model = ClassificationModel('bert', 'bert-base-cased', num_labels=3, args=model_args, use_cuda=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
print(str(model.args).replace(',', '\n'))

ClassificationArgs(adafactor_beta1=None
 adafactor_clip_threshold=1.0
 adafactor_decay_rate=-0.8
 adafactor_eps=(1e-30
 0.001)
 adafactor_relative_step=True
 adafactor_scale_parameter=True
 adafactor_warmup_init=True
 adam_betas=(0.9
 0.999)
 adam_epsilon=1e-08
 best_model_dir='outputs/best_model'
 cache_dir='cache_dir/'
 config={}
 cosine_schedule_num_cycles=0.5
 custom_layer_parameters=[]
 custom_parameter_groups=[]
 dataloader_num_workers=0
 dataset_cache_dir=None
 do_lower_case=False
 dynamic_quantize=False
 early_stopping_consider_epochs=False
 early_stopping_delta=0.01
 early_stopping_metric='eval_loss'
 early_stopping_metric_minimize=True
 early_stopping_patience=2
 encoding=None
 eval_batch_size=100
 evaluate_during_training=True
 evaluate_during_training_silent=True
 evaluate_during_training_steps=32
 evaluate_during_training_verbose=False
 evaluate_each_epoch=True
 fp16=True
 gradient_accumulation_steps=1
 learning_rate=4e-06
 local_rank=-1
 logging_steps=50
 loss_type=None
 

# Model Fine-Tuning and Evaluation

In [8]:
_, history = model.train_model(train, eval_df=dev)

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:544: UserWarning: The 'eval_df' parameter has been deprecated and will be removed in a future version. Please use 'eval_data' instead.
  warnings.warn(


  0%|          | 0/5 [00:00<?, ?it/s]

Epoch:   0%|          | 0/10 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:924: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()


Running Epoch 1 of 10:   0%|          | 0/85 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:1570: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:950: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:1570: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:950: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


0it [00:00, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:1570: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


In [10]:
# Evaluate the model
result, model_outputs, wrong_predictions = model.eval_model(dev)
result

0it [00:00, ?it/s]

Running Evaluation:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:1570: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


{'mcc': np.float64(0.9751137699101214), 'eval_loss': 0.08160129748284817}

In [11]:
predicted, probabilities = model.predict(test.text.to_list())
test['predicted'] = predicted
pd.set_option('display.max_rows', None)
print(test)

0it [00:00, ?it/s]

Predicting:   0%|          | 0/1 [00:00<?, ?it/s]

                                                text  labels  predicted
0  It took eight years for Warner Brothers to rec...       0          0
1  All the New York University students love this...       2          2
2  This Italian place is really trendy but they h...       2          2
3  In conclusion, my review of this book would be...       1          1
4  The story of this movie is focused on Carl Bra...       0          0
5  Chris O'Donnell stated that while filming for ...       0          0
6  My husband and I moved to Amsterdam 6 years ag...       2          2
7  Dame Maggie Smith performed her role excellent...       0          0
8  The new movie by Mr. Kruno was shot in New Yor...       0          0
9  I always have loved English novels, but I just...       1          1


/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:2260: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


In [12]:
print(classification_report(test['labels'], test['predicted']))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00         2
           2       1.00      1.00      1.00         3

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10



# Testing Tricky Sentences

In [14]:
difficult_sentences = [
    # Has "menu" (restaurant) but is actually about a movie
    "The director left the script on the menu at the diner.",
    # Uses food adjectives to describe a book
    "The plot of this novel was spicy, but the ending was severely undercooked.",
    # Has "reading" (book) but is about a restaurant
    "I was reading the chalkboard outside to see what the chef's specials were." ]

difficult_predictions, _ = model.predict(difficult_sentences)

# Map back to words
inverse_mapping = {0: 'movie', 1: 'book', 2: 'restaurant'}
for sentence, pred in zip(difficult_sentences, difficult_predictions):
    print(f"Prediction: {inverse_mapping[pred]:<10} | Sentence: {sentence}")

0it [00:00, ?it/s]

Predicting:   0%|          | 0/1 [00:00<?, ?it/s]

Prediction: restaurant | Sentence: The director left the script on the menu at the diner.
Prediction: book       | Sentence: The plot of this novel was spicy, but the ending was severely undercooked.
Prediction: restaurant | Sentence: I was reading the chalkboard outside to see what the chef's specials were.


/usr/local/lib/python3.12/dist-packages/simpletransformers/classification/classification_model.py:2260: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
